# Reconstructing sessions without a usable `end_user`

About a quarter of requests in the LiteLLM extracts carry no parseable session id
in `end_user`. Falling back to LiteLLM's native `session_id` does not fix this —
that column is a per-request trace id, so it turns every unparsed request into its
own one-request "session" (see `knowledge/caveats/end-user-parsing.md`).

This notebook tests whether the conversation itself can supply the missing key.
Every request resends the whole history so far, so requests from one conversation
should share an identical opening turn.

## What this notebook found

1. **The opening turn is not stable within an `end_user` session.** It is
   byte-identical across all requests in only ~20% of sessions, covering ~3% of
   requests.
2. **The reason is not client noise — it is that an `end_user` session is not one
   conversation.** 55% of consecutive request pairs inside a single session share
   *no* leading message at all. The session id identifies a client session (one
   CLI/IDE run), which contains many independent threads.
3. **Keying on the opening turn therefore recovers threads, not `end_user`
   sessions.** Scored against the session ids, it reaches ~0.95 pairwise precision
   but only ~0.24 recall — it almost never merges unrelated work, and it splits
   each session into ~11 threads, which is what a thread-level key *should* do.
4. **On the target population it works.** Of requests with no `end_user` session
   that have any user turn at all, ~70% land in a multi-request thread — against
   ~72% for requests whose session id is known. The recovered structure looks like
   the real one.

The 30% that stay singletons are mostly genuine one-shot requests, and the
requests with no user turn at all are 98% failed requests, which every metric
filters out anyway.

## Setup

Point `LLMOXIE_RAW` at a LiteLLM JSONL extract. Feature extraction streams the
file and keeps only digests and counts — no prompt text is retained, so the cache
is safe to keep on disk, though it stays gitignored regardless.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from llmoxie_analysis.analysis.session_recovery import (
    KEY_COMPONENTS,
    assign_keys,
    build_features,
    consecutive_overlap,
    partition_metrics,
)

RAW = Path(os.environ.get("LLMOXIE_RAW", "~/llmavendata/jan-feb-mar-2026-nosafemind.jsonl")).expanduser()
CACHE = Path(os.environ.get("LLMOXIE_CACHE", "cache/features.parquet"))

# Slots 1 and 2 of the validated categorical palette (dataviz skill).
BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "grid.linewidth": 0.6,
        "axes.axisbelow": True,
    }
)

In [ ]:
if CACHE.exists():
    df = pd.read_parquet(CACHE)
else:
    df = build_features(RAW)
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(CACHE, index=False)

df["has_eu_session"] = df.eu_session_id.ne("")
df["keyable"] = df.first_user_sha.ne("")
len(df), df.has_eu_session.mean()

## 1. How much of the data needs recovering

`has_eu_session` is the ground truth: these requests already have a conversation
key. The rest are what the fallback would turn into singletons.

In [ ]:
coverage = (
    df.groupby(["has_eu_session", "keyable"])
    .agg(requests=("request_id", "size"), spend=("spend", "sum"), failures=("status", lambda s: (s == "failure").sum()))
    .assign(share=lambda d: d.requests / len(df))
)
coverage.round(3)

A request with no user turn (`keyable == False`) cannot be fingerprinted at all —
there is nothing to hash. Check what those actually are before worrying about them:

In [ ]:
df[~df.keyable].status.value_counts(normalize=True).round(3)

Overwhelmingly failed requests. `knowledge/datasets/jan-mar-2026-findings.md`
already requires filtering `status == "failure"` before any metric, so these are
not a loss this method causes.

## 2. Is the opening turn stable within a session?

The hypothesis: all requests in one conversation resend the same first user turn,
so `hash(first user turn)` is a session key.

A client might append volatile context (timestamps, reminders) to its opening
turn, so test truncated prefixes too — if the instability is all in a mutable
tail, a short prefix survives it.

In [ ]:
gt = df[df.has_eu_session & df.keyable]

stability = []
for col, label in [
    ("first_user_sha", "full opening turn"),
    ("first_user_prefix_sha", "opening turn, first 2000 chars"),
]:
    per = gt.groupby("eu_session_id").agg(n=("request_id", "size"), distinct=(col, "nunique"))
    stable = per.distinct.eq(1)
    stability.append(
        {
            "anchor": label,
            "sessions_stable": int(stable.sum()),
            "sessions": len(per),
            "pct_sessions": stable.mean(),
            "pct_requests": per.loc[stable, "n"].sum() / per.n.sum(),
        }
    )
pd.DataFrame(stability).round(3)

**The hypothesis fails.** The opening turn is stable in only about a fifth of
sessions, and those sessions hold under 3% of requests. Truncating to a prefix
barely helps, which rules out "a volatile tail" as the explanation — whatever
changes, changes at the very front.

How many distinct opening turns does one session actually contain?

In [ ]:
per = gt.groupby("eu_session_id").agg(n=("request_id", "size"), distinct=("first_user_sha", "nunique"))
per[per.n > 1].distinct.describe().round(2)

## 3. Why: an `end_user` session is not one conversation

If a session were one cumulative conversation, consecutive requests would share a
long prefix of messages — request *k+1* is request *k* plus a turn or two. Measure
that directly: for each adjacent pair of requests in a session, what fraction of
the shorter history is a shared leading run?

In [ ]:
overlap = consecutive_overlap(gt, "eu_session_id")
print(f"{len(overlap)} consecutive request pairs")
print(f"  earlier history is an exact prefix of the later one: {overlap.is_continuation.mean():.1%}")
print(f"  share no leading message at all                    : {overlap.shared_fraction.eq(0).mean():.1%}")
print(f"  share 90%+ of leading messages                     : {overlap.shared_fraction.ge(0.9).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(overlap.shared_fraction, bins=40, color=BLUE)
ax.set_xlabel("shared leading messages / shorter history")
ax.set_ylabel("consecutive request pairs")
ax.set_title("Requests inside one end_user session either continue a thread or start a new one")
fig.tight_layout()

The distribution is bimodal with almost nothing in between: a pair either continues
the same thread (right spike) or shares nothing (left spike). There is no gradual
drift, which is what client noise would look like.

So `end_user.session_id` identifies a **client session** — one CLI or IDE run —
and a client session contains many independent conversations: new chats, context
resets, subagent calls, background model calls. That is the real reason the opening
turn is unstable, and it means the opening-turn key is not failing to recover
sessions; it is recovering something finer.

## 4. Scoring the key against the known session ids

Two views of the same comparison, because the two failure modes have very different
costs:

- **Pairwise precision** — of the request pairs the key grouped together, the share
  that truly belong together. Low precision means distinct conversations were
  *merged*, concatenating unrelated work. This is the dangerous error.
- **Pairwise recall** — of the pairs that truly belong together, the share the key
  found. Low recall means a session was *split*. Here splitting is expected, since
  the key targets threads.
- **Exact session rate** — the share of `end_user` sessions recovered whole, with
  nothing extra. The strictest reading of "matches the proper session id".

Pair counting is size-aware: recovering one 300-request session matters far more
than a 2-request one.

In [ ]:
variants = {
    "opening turn only": (["first_user_sha"], None),
    "+ api key alias": (["first_user_sha", "alias_sha"], None),
    "+ alias, call_type, model": (list(KEY_COMPONENTS), None),
    "+ 30 min gap split": (list(KEY_COMPONENTS), 30.0),
    "+ 120 min gap split": (list(KEY_COMPONENTS), 120.0),
    "2000-char prefix instead": (["first_user_prefix_sha", "alias_sha", "call_type"], None),
}
scores = pd.DataFrame(
    [
        partition_metrics(gt.eu_session_id, assign_keys(gt, comp, gap_minutes=gap)) | {"variant": name}
        for name, (comp, gap) in variants.items()
    ]
).set_index("variant")
scores[["n_pred_clusters", "pair_precision", "pair_recall", "pair_f1", "exact_session_rate", "n_merged"]].round(3)

Reading the table:

- **Precision ~0.89 rising to ~0.95 with a time-gap split.** The key almost never
  merges two different client sessions. The gap split is what removes most of the
  remaining merges: it separates the same opening prompt re-run on a later day.
- **Recall ~0.24.** Each session breaks into roughly eleven clusters, matching the
  thread structure found in section 3. This is not a defect of the key.
- **The 2000-char prefix is much worse** (precision ~0.53) — truncation makes
  distinct conversations collide. Hash the whole opening turn.
- **Exact session rate stays ~0.13** and should not be read as the headline. It can
  only be high if client sessions were single conversations, which they are not.

`120 min gap split` is the best variant: it beats the 30-minute split on precision
*and* recall.

## 5. Applying it to the requests that need it

In [ ]:
BEST = (list(KEY_COMPONENTS), 120.0)
un = df[~df.has_eu_session].copy()
un_keyable = un[un.keyable].copy()
un_keyable["thread"] = assign_keys(un_keyable, BEST[0], gap_minutes=BEST[1])

sizes = un_keyable.thread.value_counts()
multi = sizes[sizes > 1]
pd.Series(
    {
        "requests with no end_user session": len(un),
        "  unkeyable (no user turn, ~98% failures)": int((~un.keyable).sum()),
        "  keyable": len(un_keyable),
        "threads recovered from keyable": len(sizes),
        "sessions the native-id fallback would invent": len(un),
        "requests in a multi-request thread": int(multi.sum()),
        "share of keyable requests in a multi-request thread": round(multi.sum() / len(un_keyable), 3),
        "largest thread": int(sizes.max()),
    }
)

The success heuristic: **~70% of keyable unlabelled requests reconstruct into a
thread with more than one request**, against 3,338 singletons from the native-id
fallback.

Is 70% good? Compare it with the same key applied to requests whose session id is
known — that is the rate this method achieves on data where the structure is real.

In [ ]:
gt_threads = assign_keys(gt, BEST[0], gap_minutes=BEST[1]).value_counts()
comparison = pd.DataFrame(
    {
        "known session id": {
            "requests": len(gt),
            "threads": len(gt_threads),
            "in a multi-request thread": gt_threads[gt_threads > 1].sum() / len(gt),
            "median thread size": gt_threads.median(),
        },
        "recovered (no session id)": {
            "requests": len(un_keyable),
            "threads": len(sizes),
            "in a multi-request thread": multi.sum() / len(un_keyable),
            "median thread size": sizes.median(),
        },
    }
).T
comparison.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
bins = range(1, 22)
for series, colour, label in [(gt_threads, BLUE, "known session id"), (sizes, ORANGE, "recovered (no session id)")]:
    counts = series.clip(upper=21).value_counts(normalize=True).sort_index()
    ax.plot(counts.index, counts.to_numpy(), color=colour, linewidth=2, marker="o", markersize=4, label=label)
ax.set_xlabel("requests in thread (21 = 21 or more)")
ax.set_ylabel("share of threads")
ax.set_title("Recovered threads are distributed like real ones")
ax.legend(frameon=False)
fig.tight_layout()

The two distributions track each other closely. That is circumstantial rather than
proof — it shows the recovered grouping has the same shape as real conversation
structure, not that any individual thread is correct — but combined with 0.95
precision on the labelled half it is reasonable evidence the method is sound.

## 6. What to do with this

**Do not resurrect the native `session_id` fallback.** It invents 3,338 singleton
sessions here and inflates every per-session average.

**Use the opening-turn thread key**, as `hash(first user turn, api key alias,
call_type, model)` with a 120-minute gap split, and **record its provenance** — a
`session_id_source` column with values `end_user`, `reconstructed`, `none`. The
three populations have genuinely different reliability and analysts need to filter
on that.

**Do not mix reconstructed threads with `end_user` sessions in one count.** They
are different units: a thread is one conversation, an `end_user` session is a
client run containing ~11 of them. Mixing them makes "requests per session"
meaningless. If a single unit is needed, derive threads for *both* populations
using this key — which also makes the two halves comparable for the first time.

**Open questions worth a follow-up:**

- The ~11 threads per client session is itself a finding about how these agents
  work. Worth confirming it is real behaviour and not an artefact of the key.
- Thread-level grouping cannot be validated for the unlabelled half directly.
  The honest check is a parity run against ADLS-sourced data for an overlapping
  period, where message content is never truncated.
- 30% of keyable unlabelled requests stay singletons. Some are genuine one-shot
  calls; distinguishing those from threads the key failed to join is unresolved.